## Week Exercise

In [2]:
import pandas as pd
import planetary_computer
import pystac_client
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np


from shapely.geometry import box
from odc.stac import configure_rio, stac_load
from dask.distributed import Client, LocalCluster
from IPython.display import Image

### Part 1: Pre-Fire Baseline

In [8]:
cluster = LocalCluster(
    processes=True,
    n_workers=8,
    threads_per_worker=2,
    memory_limit="6GB",
)
client = Client(cluster)
configure_rio(cloud_defaults=True, client=client)

In [9]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
catalog.title

'Microsoft Planetary Computer STAC API'

In [10]:
bbox = [8.847198,40.193395,8.938865,40.241895] ## bbox for AOI

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2025-06-03/2025-06-03",
    query={"eo:cloud_cover": {"lt": 20}},
)

items = list(search.items())
len(items)

1

In [11]:
item = items[0]
item.id

'S2A_MSIL2A_20250603T101041_R022_T32TMK_20250603T134103'

In [12]:
item.assets

{'AOT': <Asset href=https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/32/T/MK/2025/06/03/S2A_MSIL2A_20250603T101041_N0511_R022_T32TMK_20250603T134103.SAFE/GRANULE/L2A_T32TMK_A051957_20250603T101435/IMG_DATA/R10m/T32TMK_20250603T101041_AOT_10m.tif?st=2026-01-28T15%3A45%3A25Z&se=2026-01-29T16%3A30%3A25Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-01-29T02%3A10%3A18Z&ske=2026-02-05T02%3A10%3A18Z&sks=b&skv=2025-07-05&sig=LFjvJPbdJ5Wxv74EuFmFqYMofPpSbTNVSgr0kYR%2BHj8%3D>,
 'B01': <Asset href=https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/32/T/MK/2025/06/03/S2A_MSIL2A_20250603T101041_N0511_R022_T32TMK_20250603T134103.SAFE/GRANULE/L2A_T32TMK_A051957_20250603T101435/IMG_DATA/R60m/T32TMK_20250603T101041_B01_60m.tif?st=2026-01-28T15%3A45%3A25Z&se=2026-01-29T16%3A30%3A25Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-01-29T02%3A10

In [ ]:

scl = rxr.open_rasterio(item.assets["SCL"].href).squeeze()